# Week 5: QLoRA Fine-tuning: Nemotron Nano 4B
**Goal:** Fine-tune Nano 4B on our education classification task and measure
whether it improves over the zero-shot and few-shot baselines.

**Research question:** Can fine-tuning a small model (4B) close the gap
with the larger model (30B) that currently achieves 72.85%?

**Baselines to beat:**
- Nano 4B few-shot: 70.20%
- Nano 30B reasoning OFF: 72.85% (best result so far)
- XGBoost: 70.89%

**Hardware:** FDS Dell Tower - RTX 5000 Ada 32GB

**Method:** QLoRA (Quantized Low-Rank Adaptation) via Hugging Face PEFT
- Loads 4B model in 4-bit quantization to save VRAM
- Adds small trainable LoRA adapters on top
- Only adapter weights are updated - base model stays frozen
- Training data: week3_train_5000.csv converted to chat format

## 1. Install Dependencies

In [1]:
# Run this cell once to install required packages
import subprocess
packages = [
    "transformers>=4.40.0",
    "peft>=0.10.0",
    "bitsandbytes>=0.43.0",
    "trl>=0.8.0",
    "accelerate>=0.29.0",
    "datasets>=2.18.0",
    "torch>=2.2.0",
    "scipy",
]
for pkg in packages:
    result = subprocess.run(
        ["pip", "install", pkg, "--quiet", "--break-system-packages"],
        capture_output=True, text=True
    )
    status = "OK" if result.returncode == 0 else "FAILED"
    print(f"  {status}: {pkg}")
print("Done.")

  OK: transformers>=4.40.0
  OK: peft>=0.10.0
  OK: bitsandbytes>=0.43.0
  OK: trl>=0.8.0
  OK: accelerate>=0.29.0
  OK: datasets>=2.18.0
  OK: torch>=2.2.0
  OK: scipy
Done.


## 2. Imports

In [2]:
import os
import json
import time
import re
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt

from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from trl import SFTTrainer
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch version: 2.12.0+cpu
CUDA available:  False


## 3. Configuration

In [3]:
# Model
MODEL_ID    = "nvidia/Nemotron-3-4B-Chat"  # HuggingFace model ID for Nano 4B
OUTPUT_DIR  = "../models/nano4b-qlora-education"
RESULTS_DIR = "../results"
DATA_DIR    = "../data"

os.makedirs(OUTPUT_DIR,  exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# Training config
N_TRAIN_ROWS = 1000   # rows from train set to use for fine-tuning
N_EPOCHS     = 3      # number of training epochs
BATCH_SIZE   = 4      # per device batch size
MAX_SEQ_LEN  = 512    # max tokens per sequence
LEARNING_RATE = 2e-4  # standard for QLoRA

# LoRA config
LORA_R      = 16   # rank of LoRA adapters
LORA_ALPHA  = 32   # LoRA scaling factor
LORA_DROPOUT = 0.05

print(f"Model:         {MODEL_ID}")
print(f"Train rows:    {N_TRAIN_ROWS}")
print(f"Epochs:        {N_EPOCHS}")
print(f"Batch size:    {BATCH_SIZE}")
print(f"LoRA rank:     {LORA_R}")
print(f"Output dir:    {OUTPUT_DIR}")

Model:         nvidia/Nemotron-3-4B-Chat
Train rows:    1000
Epochs:        3
Batch size:    4
LoRA rank:     16
Output dir:    ../models/nano4b-qlora-education


## 4. Load and Prepare Training Data
Convert train CSV rows to OpenAI chat format for SFT training.

In [4]:
df_train = pd.read_csv(f"{DATA_DIR}/week3_train_5000.csv")
df_test  = pd.read_csv(f"{DATA_DIR}/week3_test_5000.csv")

print(f"Train set: {len(df_train):,} rows")
print(f"Test set:  {len(df_test):,} rows")
print(f"Train label balance: {df_train['label_name'].value_counts().to_dict()}")

# Sample N_TRAIN_ROWS balanced rows from train set
train_college  = df_train[df_train["label_name"] == "college"].sample(
    N_TRAIN_ROWS // 2, random_state=42
)
train_not      = df_train[df_train["label_name"] == "not_college"].sample(
    N_TRAIN_ROWS // 2, random_state=42
)
df_finetune    = pd.concat([train_college, train_not]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\nFine-tuning sample: {len(df_finetune)} rows")
print(f"Balance: {df_finetune['label_name'].value_counts().to_dict()}")

Train set: 2,460 rows
Test set:  615 rows
Train label balance: {'not_college': 1346, 'college': 1114}

Fine-tuning sample: 1000 rows
Balance: {'not_college': 500, 'college': 500}


In [5]:
SYSTEM_PROMPT = (
    "You are an education level classifier. Given a person's demographic information, "
    "predict whether they are college-educated or not. "
    "Respond with ONLY one label: college or not_college."
)


def serialize_row(row):
    return (
        f"A {int(row['age'])}-year-old {str(row['sex']).lower().strip()}, "
        f"{str(row['marital_status']).replace('_', ' ').strip()}, "
        f"working as a {str(row['occupation']).replace('_', ' ').strip()}. "
        f"Located in {str(row['state']).strip()}."
    )


def row_to_chat(row):
    """Convert a DataFrame row to OpenAI chat format."""
    return {
        "messages": [
            {"role": "system",    "content": SYSTEM_PROMPT},
            {"role": "user",      "content": (
                f"Classify this person's education level:\n\n"
                f"{serialize_row(row)}\n\n"
                f"Answer with college or not_college only."
            )},
            {"role": "assistant", "content": row["label_name"]},
        ]
    }


# Convert to chat format
chat_data = [row_to_chat(row) for _, row in df_finetune.iterrows()]

# Show sample
print("Sample training example:")
sample = chat_data[0]
for msg in sample["messages"]:
    print(f"  [{msg['role'].upper()}] {msg['content'][:80]}")
print(f"\nTotal training examples: {len(chat_data)}")

Sample training example:
  [SYSTEM] You are an education level classifier. Given a person's demographic information,
  [USER] Classify this person's education level:

A 54-year-old male, married present, wo
  [ASSISTANT] not_college

Total training examples: 1000


## 5. Load Tokenizer and Model with 4-bit Quantization

In [6]:
print(f"Loading tokenizer from {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"Tokenizer loaded. Vocab size: {tokenizer.vocab_size:,}")
print(f"Pad token: {tokenizer.pad_token}")

Loading tokenizer from nvidia/Nemotron-3-4B-Chat...


OSError: nvidia/Nemotron-3-4B-Chat is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`

In [ ]:
# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

print(f"Loading model {MODEL_ID} in 4-bit...")
print("This may take a few minutes on first load...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

model.config.use_cache = False
model.config.pretraining_tp = 1

print(f"Model loaded.")
print(f"VRAM used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

## 6. Apply QLoRA Adapters

In [ ]:
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(model, lora_config)

# Print trainable parameter count
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable:,} ({100 * trainable / total:.2f}% of total)")
print(f"Total parameters:     {total:,}")
print(f"Frozen parameters:    {total - trainable:,}")

## 7. Prepare Dataset for Trainer

In [ ]:
def format_chat(example):
    """Format chat messages into a single text string for training."""
    text = ""
    for msg in example["messages"]:
        if msg["role"] == "system":
            text += f"<|im_start|>system\n{msg['content']}<|im_end|>\n"
        elif msg["role"] == "user":
            text += f"<|im_start|>user\n{msg['content']}<|im_end|>\n"
        elif msg["role"] == "assistant":
            text += f"<|im_start|>assistant\n{msg['content']}<|im_end|>\n"
    return {"text": text}


# Create HuggingFace dataset
hf_dataset = Dataset.from_list(chat_data)
hf_dataset = hf_dataset.map(format_chat)

# Split 90/10 train/validation
split      = hf_dataset.train_test_split(test_size=0.1, seed=42)
train_data = split["train"]
val_data   = split["test"]

print(f"Training examples:   {len(train_data)}")
print(f"Validation examples: {len(val_data)}")
print(f"\nSample formatted text:")
print(train_data[0]["text"])

## 8. Training

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=N_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=4,
    learning_rate=LEARNING_RATE,
    weight_decay=0.001,
    fp16=False,
    bf16=True,
    max_grad_norm=0.3,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=10,
    report_to="none",
    optim="paged_adamw_32bit",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_data,
    eval_dataset=val_data,
    tokenizer=tokenizer,
    args=training_args,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    packing=False,
)

print(f"Starting QLoRA fine-tuning...")
print(f"  Train examples: {len(train_data)}")
print(f"  Epochs:         {N_EPOCHS}")
print(f"  Batch size:     {BATCH_SIZE}")
print(f"  LoRA rank:      {LORA_R}")
print()

t_start = time.time()
trainer.train()
t_total = time.time() - t_start

print(f"\nTraining complete! Total time: {t_total/60:.1f} minutes")

# Save the fine-tuned adapter
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Saved adapter to: {OUTPUT_DIR}")

## 9. Training Loss Plot

In [ ]:
# Plot training loss
log_history = trainer.state.log_history
train_losses = [(x["step"], x["loss"]) for x in log_history if "loss" in x]
eval_losses  = [(x["step"], x["eval_loss"]) for x in log_history if "eval_loss" in x]

fig, ax = plt.subplots(figsize=(8, 4))
if train_losses:
    steps, losses = zip(*train_losses)
    ax.plot(steps, losses, label="Train loss", color="#378ADD")
if eval_losses:
    steps, losses = zip(*eval_losses)
    ax.plot(steps, losses, label="Val loss", color="#D85A30", marker="o")
ax.set_xlabel("Step")
ax.set_ylabel("Loss")
ax.set_title("QLoRA Fine-tuning Loss - Nano 4B")
ax.legend()
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/week5_finetune_loss.png", dpi=150)
plt.show()
print("Saved: results/week5_finetune_loss.png")

## 10. Inference on Test Set

In [ ]:
model.eval()

def predict_row(row, max_new_tokens=10):
    """Run inference on one row with the fine-tuned model."""
    prompt = (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n"
        f"Classify this person's education level:\n\n"
        f"{serialize_row(row)}\n\n"
        f"Answer with college or not_college only.<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    ).strip().lower()
    # Extract label
    if "not_college" in generated:
        return "not_college"
    elif "college" in generated:
        return "college"
    return "UNKNOWN"


print(f"Running inference on {len(df_test)} test rows...")
print(f"Estimated time: ~{len(df_test) * 0.5 / 60:.0f} minutes (much faster than Ollama)\n")

ft_results = []
t_start = time.time()

for i, row in df_test.iterrows():
    t0  = time.time()
    pred = predict_row(row.to_dict())
    ms   = round((time.time() - t0) * 1000)
    ft_results.append({
        "row_id":     i,
        "input":      serialize_row(row.to_dict()),
        "occupation": row["occupation"].replace("_", " ").strip(),
        "true_label": row["label_name"],
        "pred_label": pred,
        "correct":    pred == row["label_name"],
        "time_ms":    ms,
    })
    if (i + 1) % 100 == 0:
        elapsed   = time.time() - t_start
        done      = len(ft_results)
        remaining = (elapsed / done) * (len(df_test) - done)
        print(f"  Row {done:4d}/{len(df_test)} | Elapsed: {elapsed/60:.1f}min | Remaining: ~{remaining/60:.1f}min")

ft_df = pd.DataFrame(ft_results)
ft_df.to_csv(f"{RESULTS_DIR}/week5_finetuned_raw.csv", index=False)
print(f"\nDone! Total time: {(time.time()-t_start)/60:.1f} minutes")
print(f"Saved: results/week5_finetuned_raw.csv")
print(f"\nPred label counts: {ft_df['pred_label'].value_counts().to_dict()}")

## 11. Evaluate Fine-tuned Model

In [ ]:
label_map = {"college": 1, "not_college": 0}

valid = ft_df[ft_df["pred_label"].isin(["college", "not_college"])].copy()
unknown = len(ft_df) - len(valid)

y_true = valid["true_label"].map(label_map)
y_pred = valid["pred_label"].map(label_map)

acc    = accuracy_score(y_true, y_pred)
f1     = f1_score(y_true, y_pred, average="macro")
auc    = roc_auc_score(y_true, y_pred)
avg_ms = ft_df["time_ms"].mean()

print(f"\n{'='*50}")
print(f"  Nano 4B Fine-tuned (QLoRA)")
print(f"{'='*50}")
print(f"  Rows evaluated: {len(valid)} / {len(ft_df)}")
print(f"  Unknown:        {unknown}")
print(f"  Accuracy:       {acc:.4f}")
print(f"  Macro F1:       {f1:.4f}")
print(f"  AUC-ROC:        {auc:.4f}")
print(f"  Avg time/row:   {avg_ms:.0f}ms")
print()
print(classification_report(y_true, y_pred, target_names=["not_college", "college"]))

## 12. Final Comparison : All Models Including Fine-tuned

In [ ]:
comparison = pd.DataFrame([
    {"model": "Random Forest",             "accuracy": 0.6439, "macro_f1": 0.6400, "auc_roc": 0.6795, "week": 1},
    {"model": "XGBoost",                   "accuracy": 0.7089, "macro_f1": 0.7067, "auc_roc": 0.7447, "week": 1},
    {"model": "Nano 4B zero-shot",         "accuracy": 0.6341, "macro_f1": 0.6083, "auc_roc": 0.6156, "week": 3},
    {"model": "Nano 4B few-shot",          "accuracy": 0.7020, "macro_f1": 0.7019, "auc_roc": 0.7071, "week": 3},
    {"model": "Nano 30B zero-shot",        "accuracy": 0.7252, "macro_f1": 0.7227, "auc_roc": 0.7227, "week": 4},
    {"model": "Nano 30B reasoning OFF",    "accuracy": 0.7285, "macro_f1": 0.7266, "auc_roc": 0.7272, "week": 4},
    {"model": "Nano 4B fine-tuned (QLoRA)","accuracy": round(acc, 4), "macro_f1": round(f1, 4), "auc_roc": round(auc, 4), "week": 5},
])

print("=== FINAL COMPARISON: All Models ===")
print(comparison[["model", "accuracy", "macro_f1", "auc_roc", "week"]].to_string(index=False))

# Bar chart
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
metrics = ["accuracy", "macro_f1", "auc_roc"]
titles  = ["Accuracy", "Macro F1", "AUC-ROC"]
colors  = ["#378ADD", "#1D9E75", "#EF9F27", "#D85A30", "#8B5CF6", "#06B6D4", "#EC4899"]

for ax, metric, title in zip(axes, metrics, titles):
    vals  = comparison[metric].values
    names = [n.replace(" ", "\n") for n in comparison["model"].values]
    bars  = ax.bar(names, vals, color=colors[:len(vals)])
    ax.set_title(title)
    ax.set_ylim(0.4, 1.0)
    ax.tick_params(axis="x", rotation=15, labelsize=7)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.008,
                f"{val:.3f}", ha="center", va="bottom", fontsize=7)

plt.suptitle("Final Comparison: All Models Including Fine-tuned 4B", fontsize=11)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/week5_final_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: results/week5_final_comparison.png")

# Save comparison
comparison.to_csv(f"{RESULTS_DIR}/week5_final_summary.csv", index=False)
print("Saved: results/week5_final_summary.csv")

## 13. Week 5 Fine-tuning Summary

Fill in after running all cells:

| Model | Accuracy | Macro F1 | AUC-ROC | Notes |
|---|---|---|---|---|
| Nano 4B zero-shot | 63.41% | 0.6083 | 0.6156 | No training |
| Nano 4B few-shot | 70.20% | 0.7019 | 0.7071 | 5 examples |
| Nano 30B reasoning OFF | 72.85% | 0.7266 | 0.7272 | Best so far |
| Nano 4B fine-tuned (QLoRA) | ___ | ___ | ___ | 1000 rows, 3 epochs |

**Key question:** Did fine-tuning the 4B model close the gap with the 30B?

**Training config used:**
- Training rows: 1000 (balanced 500 college + 500 not_college)
- Epochs: 3
- LoRA rank: 16
- Learning rate: 2e-4
- Hardware: FDS RTX 5000 Ada 32GB